In [0]:
from pyspark.sql import functions as F

# 1. Pipeline Paths (Matching your style)
landing_root = "/Volumes/dev/kruger/landing/raw_youtube"
staging_root = "/Volumes/dev/kruger/staging/filtered_youtube"

print(f"Processing raw YouTube metrics from {landing_root} into {staging_root}")

try:
    # 2. Batch read the raw JSON file (We use multiline because YouTube files have indents)
    df_raw = spark.read.option("multiline", "true").json(landing_root)
    
    # 3. Pull out EXACTLY the explicit fields you requested
    df_filtered = df_raw.select(F.explode("items").alias("video")).select(
        F.col("video.snippet.title").alias("title"),
        F.col("video.id").alias("id"),
        F.col("video.snippet.publishedAt").alias("published_at"),
        F.col("video.snippet.liveBroadcastContent").alias("live_broadcast_content"),
        F.col("video.statistics.viewCount").alias("view_count"),
        F.col("video.statistics.commentCount").alias("comment_count"),
        F.col("video.statistics.favoriteCount").alias("favorite_count"),
        F.col("video.statistics.likeCount").alias("like_count")
    )
    
    # 4. Save directly as a clean, simple JSON file in the Staging Volume
    df_filtered.write.mode("overwrite").json(staging_root)
    
    # 5. Print a row count summary to confirm success (just like your template style)
    count = spark.read.json(staging_root).count()
    print("-" * 45)
    print(f"{'source':<20}{'format':<8}{'row_count'}")
    print("-" * 45)
    print(f"{'latest_sightings':<20}{'json':<8}{count}")

except Exception as e:
    print(f"Error processing batch: {e}")
